In [1]:
import pandas as pd
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import os
import ssl
from pathlib import Path

In [2]:
# Disable SSL verification at the environment level
os.environ['PYTHONHTTPSVERIFY'] = '0'
os.environ['REQUESTS_CA_BUNDLE'] = ''
os.environ['CURL_CA_BUNDLE'] = ''
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

# Monkey-patch SSL to skip certificate verification
ssl._create_default_https_context = ssl._create_unverified_context

# Also patch httpx which huggingface_hub uses
import httpx
_original_client_init = httpx.Client.__init__

def _patched_client_init(self, *args, **kwargs):
    kwargs.setdefault('verify', False)
    _original_client_init(self, *args, **kwargs)

httpx.Client.__init__ = _patched_client_init

In [3]:
# Dynamically find project root by looking for a marker file/folder
# Works regardless of who runs it or where the repo is cloned
NOTEBOOK_DIR = Path.cwd()

# Walk up until we find the project root (identified by pyproject.toml, .git, or src folder)
PROJECT_ROOT = NOTEBOOK_DIR
for parent in [NOTEBOOK_DIR] + list(NOTEBOOK_DIR.parents):
    if (parent / '.git').exists() or (parent / 'pyproject.toml').exists():
        PROJECT_ROOT = parent
        break

DATA_DIR = PROJECT_ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)
(DATA_DIR / 'queries').mkdir(exist_ok=True)

os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {DATA_DIR}")

Project root: c:\Users\tremopoulosa\OneDrive - Reed Elsevier Group ICO Reed Elsevier Inc\Documents\code\writing-coach-eval
Data dir: c:\Users\tremopoulosa\OneDrive - Reed Elsevier Group ICO Reed Elsevier Inc\Documents\code\writing-coach-eval\data


OpenRewriteEval dataset

In [4]:
df_ore = pd.read_parquet("hf://datasets/gabrielmbmb/OpenRewriteEval/data/train-00000-of-00001.parquet")

c:\Users\tremopoulosa\OneDrive - Reed Elsevier Group ICO Reed Elsevier Inc\Documents\code\writing-coach-eval\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
df_ore.head()

,source,target,comment,task
0,Caught red handed by the pedestrian’s roadbloc...,I was impeded by the sight of a red hand of th...,write a story about a person who is always late,others
1,If there were any suggestions of Mendelssohn i...,If there were any hints of Mendelssohn lurking...,write in style of roger ebert,others
2,Deep Tissue Massage is a form of bodywork that...,Deep Tissue Massage soothes tension in the bod...,add a call to action,others
3,The procedure for constructing your resume and...,The procedures of building our resume through ...,make it more friendly and personal,others
4,A. Action of the Board: The Board upon a findi...,A. Action of the Board:\n-If the Board determi...,Use bullet points to make this more readable,others


In [7]:
df_ore.drop_duplicates(subset=['comment'], inplace=True)
df_ore.shape

(1033, 4)

In [8]:
df_ore['task'].value_counts()

task
wiki          560
others        338
elaborate     102
formality      12
shorten        12
paraphrase      9
Name: count, dtype: int64

In [11]:
distinct_queries = df_ore['comment'].to_list()
with open('data/queries/distinct_queries_ore.txt', 'w+') as f:
    
    for items in distinct_queries:
        f.write('%s\n' %items)

f.close()

In [12]:
queries_revise_simple = [
    "Fix grammatical errors and sentence structures.",
    "Correct grammar and spelling, improve style, cohesion, and tone.",
    "Make sure the text is clear and easy to understand",
    "Make the text more readable and comprehensive. Remove jargon.",
    "write in a more formal and professional style",
    "Use a more conversational tone.",
    "Use bullet points to make this more readable",
    "Reorganize the information.",
    "Rewrite the text to be neutral.",
    "make it more objective and less biased",
    "Paraphrase this text"
]

queries_revise_research = [
    "Copyedit to improve the text.",
    "add more supporting evidence",
    "add more examples to support your argument",
    "add references",
    "add more facts and statistics",
    "make it more persuasive with stronger arguments",
    "add a counter-argument",
    "add more details",
]

queries_respond = [
    "Elaborate and write your opinion on the topic",
    "write a set of guidelines for new copyeditors",
    "create a list of questions",
    "write a persuasive argument for or against the text",
]

In [13]:
df_ore.rename(columns={'source': 'input', 'comment': 'query'}, inplace=True)
df_ore.reset_index(drop=True, inplace=True)

In [14]:
df_ore_revise_simple = df_ore[df_ore['query'].isin(queries_revise_simple)]
df_ore_revise_research = df_ore[df_ore['query'].isin(queries_revise_research)]
df_ore_respond = df_ore[df_ore['query'].isin(queries_respond)]

In [15]:
df_ore_sample = pd.concat([
    df_ore_revise_simple.assign(dataset='OpenRewriteEval', route='REVISE_SIMPLE'), 
    df_ore_revise_research.assign(dataset='OpenRewriteEval', route='REVISE_RESEARCH'),
    df_ore_respond.assign(dataset='OpenRewriteEval', route='RESPOND')
    ]).reset_index(drop=True)
print(df_ore_sample.shape)
df_ore_sample.head()

(23, 6)


,input,target,query,task,dataset,route
0,A. Action of the Board: The Board upon a findi...,A. Action of the Board:\n-If the Board determi...,Use bullet points to make this more readable,others,OpenRewriteEval,REVISE_SIMPLE
1,This book consists of magazine and newspaper a...,This text has articles from both magazines and...,Make sure the text is clear and easy to unders...,others,OpenRewriteEval,REVISE_SIMPLE
2,The Board remains concerned about the challeng...,"Hello, there! So, the Board is not worried abo...",Use a more conversational tone.,others,OpenRewriteEval,REVISE_SIMPLE
3,"Last, we like to see a “buying thrust” at majo...","As the last part of the report, we would like ...",write in a more formal and professional style,others,OpenRewriteEval,REVISE_SIMPLE
4,"The situation, however, was different in 2004,...","In 2004, the JVP, the Janatha Vimukthi Peramun...",make it more objective and less biased,others,OpenRewriteEval,REVISE_SIMPLE


Coedit dataset

In [16]:
splits = {'train': 'train.jsonl', 'validation': 'validation.jsonl'}
df_coedit = pd.read_json("hf://datasets/grammarly/coedit/" + splits["train"], lines=True)

In [17]:
df_coedit.head()

,_id,task,src,tgt
0,1,gec,Remove all grammatical errors from this text: ...,"For example, countries with a lot of deserts c..."
1,2,gec,Improve the grammaticality: As the number of p...,"As the number of people grows, the need for a ..."
2,3,gec,Improve the grammaticality of this sentence: B...,Besides some technological determinists that a...
3,4,gec,Remove all grammatical errors from this text: ...,Safety is one of the crucial problems that man...
4,5,gec,Fix grammaticality in this sentence: On one ha...,"On the one hand, more and more viruses and hac..."


In [18]:
df_coedit['task'].value_counts()

task
gec               19823
paraphrase        15370
simplification    11440
coherence         10616
neutralize        10570
clarity            1252
Name: count, dtype: int64

In [19]:
df_coedit['query'] = df_coedit['src'].apply(lambda x: x.split(':')[0])
df_coedit['input'] = df_coedit['src'].apply(lambda x: x.split(':')[1])
df_coedit.drop_duplicates(subset=['query'], inplace=True)
print(df_coedit.shape)
df_coedit['task'].value_counts()

(124, 6)


task
gec               27
clarity           24
coherence         22
neutralize        19
simplification    18
paraphrase        14
Name: count, dtype: int64

In [20]:
df_coedit.rename(columns={'tgt': 'target'}, inplace=True)
df_coedit.drop(columns=['src', '_id'], inplace=True)
df_coedit.reset_index(drop=True, inplace=True)
df_coedit.head()

,task,target,query,input
0,gec,"For example, countries with a lot of deserts c...",Remove all grammatical errors from this text,"For example, countries with a lot of deserts ..."
1,gec,"As the number of people grows, the need for a ...",Improve the grammaticality,"As the number of people grows, the need of ha..."
2,gec,Besides some technological determinists that a...,Improve the grammaticality of this sentence,Besides some technologically determinists tha...
3,gec,"On the one hand, more and more viruses and hac...",Fix grammaticality in this sentence,On one hand more and more virus and hack can ...
4,gec,We can have a fingerprint verification reader ...,Improve the grammar of this text,We can have fingerprint verification reader c...


In [21]:
distinct_queries = df_coedit['query'].to_list()

In [22]:
queries_to_keep = [
'Fix disfluencies in the sentence'
]

In [23]:
df_coedit_sample = df_coedit[df_coedit['query'].isin(queries_to_keep)]
print(df_coedit_sample.shape)
df_coedit_sample.head()

(1, 4)


,task,target,query,input
10,gec,"An example is water, which is a good renewable...",Fix disfluencies in the sentence,An example is water which is a good renewable...


In [24]:
df_coedit_sample = pd.concat([
    df_coedit_sample.assign(dataset='CoEdiT', route='REVISE_SIMPLE'),
    ]).reset_index(drop=True)
print(df_coedit_sample.shape)
df_coedit_sample.head()

(1, 6)


,task,target,query,input,dataset,route
0,gec,"An example is water, which is a good renewable...",Fix disfluencies in the sentence,An example is water which is a good renewable...,CoEdiT,REVISE_SIMPLE


Kiwi dataset

In [25]:
df_kiwi = pd.read_json("hf://datasets/fangyuan/kiwi/interaction_data_with_annotator_ids.jsonl", lines=True)
df_kiwi.head()

,original_question,model_name,initial_answer,initial_docs,interaction,annotator_id
0,What are the prior approaches proposed to impr...,gpt-4,Prior approaches to improve the faithfulness o...,[{'text': 'As expert systems are brittle and c...,[{'instruction': 'What are specific examples o...,0
1,How have contrastive learning techniques been ...,gpt-3.5-turbo,Contrastive learning techniques have been appl...,[{'text': 'Among possible unsupervised learnin...,[{'instruction': 'In the first couple of sente...,1
2,How have contrastive learning techniques been ...,gpt-4,Contrastive learning techniques have been used...,[{'text': 'Among possible unsupervised learnin...,[{'instruction': 'How has contrastive learning...,0
3,What are the different approaches proposed for...,gpt-4,"The paper ""Query Reformulation using Query His...",[{'text': 'Query reformulation (QR) has proven...,"[{'instruction': 'Add short definitions of ""qu...",2
4,How has active learning been used for semantic...,gpt-4,Active learning has been used in semantic pars...,[{'text': 'We have discussed practical active ...,[{'instruction': 'Keep the text as it is but a...,2


In [26]:
# Create triplets: (input, query, target) from Kiwi interactions
triplets = []

for _, row in df_kiwi.iterrows():
    interaction_list = row['interaction']
    if not isinstance(interaction_list, list) or len(interaction_list) == 0:
        continue
    
    for turn_idx, turn in enumerate(interaction_list):
        if not isinstance(turn, dict):
            continue
        
        instruction = turn.get('instruction')
        answer_1 = turn.get('answer_1')
        answer_2 = turn.get('answer_2')
        rating = turn.get('rating')
        
        if not instruction:
            continue
        
        # Determine input: initial_answer for turn 0, previous turn's output for turn > 0
        if turn_idx == 0:
            current_input = row['initial_answer']
        else:
            prev_turn = interaction_list[turn_idx - 1]
            prev_answer_2 = prev_turn.get('answer_2')
            prev_answer_1 = prev_turn.get('answer_1')
            current_input = prev_answer_2 if prev_answer_2 and prev_answer_2 != '' else prev_answer_1
        
        # Determine target: prefer answer_2 over answer_1
        target = answer_2 if answer_2 and answer_2 != '' else answer_1
        
        if not current_input or not target:
            continue

        triplets.append({
            'input': current_input,
            'query': instruction,
            'target': target,
            'rating': rating,
            'turn_number': turn_idx,
            'model_name': row.get('model_name'),
            'has_answer_2': bool(answer_2 and answer_2 != '')
        })

df_kiwi_triplets = pd.DataFrame(triplets)

print(f"Created {len(df_kiwi_triplets)} triplets from {len(df_kiwi)} rows")
# print(f"\nTurn distribution:")
# print(df_kiwi_triplets['turn_number'].value_counts().sort_index())
# print(f"\nRating distribution:")
# print(df_kiwi_triplets['rating'].value_counts())
# print(f"\nRows with user-edited answer_2: {df_kiwi_triplets['has_answer_2'].sum()}")

df_kiwi_triplets.head()

Created 1259 triplets from 234 rows


,input,query,target,rating,turn_number,model_name,has_answer_2
0,Prior approaches to improve the faithfulness o...,What are specific examples of the tasks involv...,Previous approaches to improve the faithfulnes...,bad,0,gpt-4,False
1,Previous approaches to improve the faithfulnes...,What are the strong results that these models ...,Prior approaches to improve the faithfulness o...,good,1,gpt-4,False
2,Prior approaches to improve the faithfulness o...,Which specific tasks have these models been ap...,Prior approaches to enhance the faithfulness o...,good,2,gpt-4,False
3,Prior approaches to enhance the faithfulness o...,Why do these models only slightly outperform t...,Prior approaches to improve the faithfulness o...,bad,3,gpt-4,False
4,Prior approaches to improve the faithfulness o...,How much better are these models than the base...,Previous strategies aimed at improving the fai...,neutral,4,gpt-4,True


In [27]:
print(df_kiwi_triplets.shape)
df_kiwi_triplets.drop_duplicates(subset=['query'], inplace=True)
df_kiwi_triplets.shape

(1259, 7)


(1245, 7)

In [ ]:
distinct_queries = df_kiwi_triplets['query'].to_list()
with open('data/queries/distinct_queries_kiwi.txt', 'w+') as f:
    
    for items in distinct_queries:
        f.write('QUERY:%s\n' %items)

f.close()

In [ ]:
# Read the file and join multi-line queries
with open('data/queries/distinct_queries_kiwi.txt', 'r', encoding='utf-8') as f:
    content = f.read()

# Split by 'QUERY:' and filter out empty strings
queries = [q.strip() for q in content.split('QUERY:') if q.strip()]

# Write back with each query on a single line
with open('data/queries/distinct_queries_kiwi2.txt', 'w', encoding='utf-8') as f:
    for query in queries:
        # Replace newlines within the query with spaces
        clean_query = ' '.join(query.split())
        f.write(f'{clean_query}\n')

In [30]:
queries_revise_simple = [
    "Rewrite the answer to be concise and directly answer the question. Try not to delete any of the content, just make it a bit more concise."
]

queries_revise_research = [
    "Can you provide more evidence for why in-context learning works spanning multiple works and what they've shown?",
    "Include more methods that have been tried, maybe starting with simpler methods that lead to more complex ones to help orient the reader and introduce more complex concepts iteratively.",
    "Add a few sentences at the beginning to briefly explain the area and motivate why this problem is important.",
    "In the first couple of sentences, provide a general definition of contrastive learning and explain why it is useful in building foundation models.",
    "As first sentence of the text, add a short definition of \"table question answering\""
    ]

queries_respond = [
    "Summarize and answer the question directly. When summarizing, try to put methods together into larger categories for easier reading.",
    "What are the strong results that these models produce for mathematical reasoning and programmatic execution tasks?",
    "Which specific tasks have these models been applied to?",
    "How much better are these models than the baselines?",
    "What is the performance of the models using contrastive learning?",
    "How well do state-of-the-art models perform on these datasets?",
    "What metric was used to measure performance on these datasets?",
    "What are advantages and disadvantages for each method listed?",
    "How does the selection of the pretraining corpus affect the performance and output of language models?"
]

queries_research = [
    "Find every paper related to task-specific pre-training adaptation and include the methods mentioned there in your answer. Be exhaustive in your list.",
    "Find a bunch of papers that describe different methods. Take them and group them based on similarity. For each group, give it a name, high level description, then paper specific details.",
    "Find many more papers and use them to form groups that describe the different factors. For each group, give it a name and a description of the group. At the end of each group should be the list of paper references.",
    "Find more parameter efficient tuning methods and group them by their similarities rather than enumerating them one by one. Try to find their big similarities and differences to give the reader a comprehensive overview of all the types of all parameter efficient tuning methods.",
    "Find more methods from different papers. Look at reasoning skills. Or how textual + tabular data pretraining can help. Weak supervision may be a good source to include as well. Find more papers to add to the groups and add new groups as needed.",
    "Find more ways that human feedback was used, then group them into categories with a title and high-level description of what it is.",
    "Can you retrieve information about additional input perturbation-based methods from supporting papers or passages?",
    "Are there other ways of doing task-specific pretraining mentioned in the supporting papers or passages?",
    "Cool! Are there any other existing methods for leveraging information in knowledge bases for task-specific language model fine-tuning?",
    "Can you list additional approaches proposed for query expansion and reformulation?",
    "Can you add more information discussing empirical observations about in-context learning behavior to this answer?",
    "Retrieve \"Compressing deep neural networks by matrix product operators\" and summarize how it answers the question.",
    "Retrieve \"Tensorizing Neural Networks\" and summarize how just that paper answers the question"
]

In [31]:
df_kiwi_triplets.drop(columns=['rating', 'turn_number', 'model_name', 'has_answer_2'], inplace=True)

In [32]:
def find_matching_queries(target_queries, df_queries, threshold=90):
    """Find queries in dataframe using exact match first, then fuzzy matching."""
    matched_queries = []
    unmatched = []
    
    for target in target_queries:
        # Try exact match first
        if target in df_queries:
            matched_queries.append(target)
        else:
            # Try fuzzy match
            match = process.extractOne(target, df_queries, scorer=fuzz.ratio)
            if match and match[1] >= threshold:
                matched_queries.append(match[0])
                # print(f"Fuzzy matched:\n  Target: {target[:80]}...\n  Found:  {match[0][:80]}...\n  Score: {match[1]}\n")
            else:
                unmatched.append(target)
                # print(f"NO MATCH for: {target[:80]}...")
    
    return matched_queries, unmatched

# Get all unique queries from dataset
all_df_queries = df_kiwi_triplets['query'].unique().tolist()

# Find matches for each category
# print("=== REVISE SIMPLE ===")
matched_revise_simple, unmatch_rs = find_matching_queries(queries_revise_simple, all_df_queries)

# print("\n=== REVISE RESEARCH ===")
matched_revise_research, unmatch_rr = find_matching_queries(queries_revise_research, all_df_queries)

# print("\n=== RESPOND ===")
matched_respond, unmatch_resp = find_matching_queries(queries_respond, all_df_queries)

# print("\n=== RESEARCH ===")
matched_research, unmatch_res = find_matching_queries(queries_research, all_df_queries)

# Create filtered dataframes with matched queries
df_kiwi_revise_simple = df_kiwi_triplets[df_kiwi_triplets['query'].isin(matched_revise_simple)]
df_kiwi_revise_research = df_kiwi_triplets[df_kiwi_triplets['query'].isin(matched_revise_research)]
df_kiwi_respond = df_kiwi_triplets[df_kiwi_triplets['query'].isin(matched_respond)]
df_kiwi_research = df_kiwi_triplets[df_kiwi_triplets['query'].isin(matched_research)]

print(f"\n=== RESULTS ===")
print(f"Revise Simple: {len(df_kiwi_revise_simple)} rows")
print(f"Revise Research: {len(df_kiwi_revise_research)} rows")
print(f"Respond: {len(df_kiwi_respond)} rows")
print(f"Research: {len(df_kiwi_research)} rows")


=== RESULTS ===
Revise Simple: 1 rows
Revise Research: 5 rows
Respond: 9 rows
Research: 13 rows


In [33]:
df_kiwi_sample = pd.concat([
    df_kiwi_revise_simple.assign(dataset='Kiwi', route='REVISE_SIMPLE'), 
    df_kiwi_revise_research.assign(dataset='Kiwi', route='REVISE_RESEARCH'),
    df_kiwi_respond.assign(dataset='Kiwi', route='RESPOND'),
    df_kiwi_research.assign(dataset='Kiwi', route='RESEARCH')
    ]).reset_index(drop=True)
print(df_kiwi_sample.shape)
df_kiwi_sample.head()

(28, 5)


,input,query,target,dataset,route
0,Cognition-based factors have been utilized as ...,Rewrite the answer to be concise and directly ...,Cognition-based factors have been utilized as ...,Kiwi,REVISE_SIMPLE
1,Contrastive learning techniques have been appl...,"In the first couple of sentences, provide a ge...",Contrastive learning is a technique that aims ...,Kiwi,REVISE_RESEARCH
2,Diffusion models can be applied to domains wit...,"Include more methods that have been tried, may...",Diffusion models can be applied to domains wit...,Kiwi,REVISE_RESEARCH
3,Different techniques have been developed to as...,Add a few sentences at the beginning to briefl...,Automated evaluation techniques for neural tex...,Kiwi,REVISE_RESEARCH
4,In-context learning is an emergent ability of ...,Can you provide more evidence for why in-conte...,In-context learning works due to the LM's abil...,Kiwi,REVISE_RESEARCH


Concatenate datasets

In [34]:
# Find common columns across all dataframes
common_cols = list(set(df_ore_sample.columns) & set(df_coedit_sample.columns) & set(df_kiwi_sample.columns))

# Concatenate using only common columns
df_combined = pd.concat([
    df_ore_sample[common_cols], 
    df_coedit_sample[common_cols],
    df_kiwi_sample[common_cols],
], ignore_index=True)

print(f"Common columns: {common_cols}")
print(f"Combined shape: {df_combined.shape}")
df_combined.head()

Common columns: ['query', 'dataset', 'input', 'route', 'target']
Combined shape: (52, 5)


,query,dataset,input,route,target
0,Use bullet points to make this more readable,OpenRewriteEval,A. Action of the Board: The Board upon a findi...,REVISE_SIMPLE,A. Action of the Board:\n-If the Board determi...
1,Make sure the text is clear and easy to unders...,OpenRewriteEval,This book consists of magazine and newspaper a...,REVISE_SIMPLE,This text has articles from both magazines and...
2,Use a more conversational tone.,OpenRewriteEval,The Board remains concerned about the challeng...,REVISE_SIMPLE,"Hello, there! So, the Board is not worried abo..."
3,write in a more formal and professional style,OpenRewriteEval,"Last, we like to see a “buying thrust” at majo...",REVISE_SIMPLE,"As the last part of the report, we would like ..."
4,make it more objective and less biased,OpenRewriteEval,"The situation, however, was different in 2004,...",REVISE_SIMPLE,"In 2004, the JVP, the Janatha Vimukthi Peramun..."


In [38]:
df_reordered = df_combined.iloc[:, [1, 3, 0, 2, 4]]
df_reordered.head()

,dataset,route,query,input,target
0,OpenRewriteEval,REVISE_SIMPLE,Use bullet points to make this more readable,A. Action of the Board: The Board upon a findi...,A. Action of the Board:\n-If the Board determi...
1,OpenRewriteEval,REVISE_SIMPLE,Make sure the text is clear and easy to unders...,This book consists of magazine and newspaper a...,This text has articles from both magazines and...
2,OpenRewriteEval,REVISE_SIMPLE,Use a more conversational tone.,The Board remains concerned about the challeng...,"Hello, there! So, the Board is not worried abo..."
3,OpenRewriteEval,REVISE_SIMPLE,write in a more formal and professional style,"Last, we like to see a “buying thrust” at majo...","As the last part of the report, we would like ..."
4,OpenRewriteEval,REVISE_SIMPLE,make it more objective and less biased,"The situation, however, was different in 2004,...","In 2004, the JVP, the Janatha Vimukthi Peramun..."


In [36]:
df_reordered.sort_values(by=['route', 'dataset'], inplace=True)
df_reordered.reset_index(drop=True, inplace=True)
df_reordered['route'].value_counts()

route
RESEARCH           13
RESPOND            13
REVISE_RESEARCH    13
REVISE_SIMPLE      13
Name: count, dtype: int64

In [37]:
df_reordered.to_csv("data/data_routes_original.csv", index=False)

In [39]:
pd.set_option('display.max_colwidth', None)
df_reordered.head(52)

,dataset,route,query,input,target
0,OpenRewriteEval,REVISE_SIMPLE,Use bullet points to make this more readable,"A. Action of the Board: The Board upon a finding by a majority vote that a member or chapter has violated the Articles of Incorporation or Bylaws shall notify the member or chapter of its delinquency in writing. If the delinquency is subject to being remedied, reasonable opportunity within the discretion of the Board for correction shall be allowed if however the breech is not subject to being corrected or is not corrected, the Board may by two-third (2/3) majority vote terminate or suspend for up to one (01) year. An individual member or chapter termination or Suspension by the Board does not become effective until that entire act has been affirmed by the four-fifths (4/5) vote of the Membership at the Annual or Special Meeting at which a quorum is present. In the event of a resolution calling for the termination of suspension of a member or chapter, the Board shall forthwith call a special meeting of the Membership.","A. Action of the Board:\n-If the Board determines by a majority vote that a member or chapter has violated the Articles of Incorporation or Bylaws, the Board shall inform the member or chapter of its negligence in writing.\n-If the delinquency is remedial, the Board shall provide a reasonable opportunity within its discretion.\n-However, if the breech is not remedial or is not corrected, the Board may, by a two-thirds (2/3) majority vote, terminate or suspend for up to a year.\n-The termination or suspension of an individual member or chapter by the Board is not effective until it is affirmed by a four-fifths (4/5) vote of the membership at the annual or special meeting, where a quorum is present.\n-In situations where there is a resolution calling for the termination of the suspension of a member or chapter, the Board will convene a special meeting of the membership."
1,OpenRewriteEval,REVISE_SIMPLE,Make sure the text is clear and easy to understand,"This book consists of magazine and newspaper articles, interviews, record reviews, excerpts from books, and other material related to the history of The Doors, to whose memory it is dedicated.This book consists of magazine and newspaper articles, interviews, record reviews, excerpts from books, and other material related to the history of The Doors, to whose memory it is dThis book consists of magazine and newspaper articles, interviews, record reviews, excerpts from books, and other material related to the history of The Doors, to whose memory it is dedicated.This book consists of magazine and newspaper articles, interviews, record reviews, excerpts from books, and other material related to the history of The Doors, to whose memory it is dedicated.","This text has articles from both magazines and newspapers, recorded consultations, record reviews, parts from other texts, and more resources involved with The Doors’ history, to who this text dedicated it’s memory toward."
2,OpenRewriteEval,REVISE_SIMPLE,Use a more conversational tone.,"The Board remains concerned about the challenge of funding the deficit of approximately £35m from 2017/18. An application has been made to Government for a special dispensation to capitalise the deficit. If granted, the plan is to apply capital receipts to fund the deficit. The Board was strongly supportive of this application as it was the best available course of action in the circumstances. The Board is sceptical about whether the alternative option of making further cuts in 2018/19 is achievable in the time available, and is concerned about potentially irreparable damage to services and impact on residents. However, the Board noted that Government support for such a dispensation was unlikely to be forthcoming unless the County Council was seen to be taking effective action to avoid a further deficit in 2018/19.","Hello, there! So, the Board is not worried about the issue of funding the deficit of approximately £35m from

In [40]:
df_reordered[['dataset', 'route', 'query']].head(52)

,dataset,route,query
0,OpenRewriteEval,REVISE_SIMPLE,Use bullet points to make this more readable
1,OpenRewriteEval,REVISE_SIMPLE,Make sure the text is clear and easy to understand
2,OpenRewriteEval,REVISE_SIMPLE,Use a more conversational tone.
3,OpenRewriteEval,REVISE_SIMPLE,write in a more formal and professional style
4,OpenRewriteEval,REVISE_SIMPLE,make it more objective and less biased
5,OpenRewriteEval,REVISE_SIMPLE,Reorganize the information.
6,OpenRewriteEval,REVISE_SIMPLE,Make the text more readable and comprehensive. Remove jargon.
7,OpenRewriteEval,REVISE_SIMPLE,Rewrite the text to be neutral.
8,OpenRewriteEval,REVISE_SIMPLE,Fix grammatical errors and sentence structures.
9,OpenRewriteEval,REVISE_SIMPLE,"Correct grammar and spelling, improve style, cohesion, and tone."


Check synthetic data created from synthesize_domain_data.py script


In [11]:
df_synth = pd.read_csv("data/data_routes_synthetic.csv")
print(df_synth.shape)
pd.set_option('display.max_colwidth', None)

(156, 5)


Check synthetic data for REVISE_SIMPLE ROUTE 

In [13]:
# df_synth[df_synth['route'] == 'REVISE_SIMPLE'].head(50)

In [18]:
df_synth['query'][df_synth['route'] == 'REVISE_SIMPLE'].head(52)

81     Rewrite the explanation to be concise and directly answer the question. Try not to delete any of the content, just make it a bit more concise.
82          Rewrite the answer to be concise and directly answer the question. Try not to delete any of the content, just make it a bit more concise.
83          Rewrite the answer to be concise and directly answer the question. Try not to delete any of the content, just make it a bit more concise.
120                                                                                                  Fix grammar and punctuation throughout the text.
121                                                                                   Simplify the language to make it easier for a general audience.
122                                                                                                        Shorten and make the passage more concise.
123                                                                                                 

RESPOND

In [19]:
# df_synth[df_synth['route'] == 'RESPOND'].head(50)
df_synth['query'][df_synth['route'] == 'RESPOND'].head(52)

39                      What are the strong results that these models produce for predicting particle interactions and simulating quantum dynamics?
40                             What are the strong results that these models produce for enzymatic kinetics and metabolic pathway simulation tasks?
41    What are the strong results that these theories produce for interpreting complex cultural narratives and performing discourse analysis tasks?
42                                                                                    Which specific experiments have these models been applied to?
43                                                                  Which specific biochemical processes have these enzyme systems been applied to?
44                                                       Which specific areas of social inquiry have these interpretive frameworks been applied to?
45                                                                                             How much better a

REVISE_RESEARCH

In [20]:
# df_synth[df_synth['route'] == 'REVISE_RESEARCH'].head(50)
df_synth['query'][df_synth['route'] == 'REVISE_RESEARCH'].head(52)


66                                                                                                                               In the first couple of sentences, provide a general definition of quantum entanglement and explain why it is useful in developing quantum communication systems.
67                                                                                                                                In the first couple of sentences, provide a general definition of enzyme allosteric regulation and explain why it is useful in understanding metabolic control.
68                                                                                                                            In the first couple of sentences, provide a general definition of cultural hybridity and explain why it is useful in understanding postcolonial identity formation.
69                                                                       Include more experimental or theoretical methods that hav

RESEARCH

In [21]:
# df_synth[df_synth['route'] == 'RESEARCH'].head(50)
df_synth['query'][df_synth['route'] == 'RESEARCH'].head(52)

0                                                                                                                                                                                                                                      Retrieve 'Tensor Networks in Quantum Many-Body Physics' and summarize how just that paper answers the question
1                                                                                                                                                                                                                                          Retrieve "Allosteric Regulation in Enzyme Kinetics" and summarize how just that paper answers the question
2                                                                                                                                                                                                                                                               Retrieve "Imagined Communities" and summarize how just that 